# Training visualization

Read exact five-seed training logs and save five figures and selected-checkpoint tables. No checkpoints, weights, Torch or GPU are required. Use a fresh kernel and run cells in order. Bands show sample SD across training seeds, not confidence intervals.


## 1. Parameters
Choose exact run paths and manifest hashes. Reuse retains numerical results and needs a new presentation ID.


In [12]:
from pathlib import Path
# Directory containing the five model directories; on Kaggle use the actual dataset mount.
# TRAINING_ROOT = Path("")
# OUTPUT_PROJECT_ROOT = Path("/kaggle/working")

def find_project_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "src" / "phonetics").is_dir():
            return candidate
        child = candidate / "project"
        if (child / "src" / "phonetics").is_dir():
            return child
    raise FileNotFoundError("Could not find project root")

PROJECT_ROOT = find_project_root()

TRAINING_ROOT = PROJECT_ROOT / 'runs' / 'duyu-chinese-lessons-zh-cn-v1'
OUTPUT_PROJECT_ROOT = PROJECT_ROOT
TRAINING_RUNS = {
    'syllable': {"path": TRAINING_ROOT / 'syllable' / '20261029T094823Z-3b73a567', "manifest_sha256": '131fe3e90ac19e81ec3c289e8dee2f0978e853c9be06b6056cb3f9dbafd4e911'},
    'components': {"path": TRAINING_ROOT / 'components' / '20261029T131437Z-7a51d166', "manifest_sha256": 'ec3f5e31aaa5a833f8d9fd30b6b11c3c7ee91ac9e7b757016e23a2f21e67b935'},
    'acoustic': {"path": TRAINING_ROOT / 'acoustic' / '20261029T131638Z-df3de704', "manifest_sha256": '005033c577dbd06b5ec3afc2da1ab9b787d9fe7b71b815cbde65ac0ad48f3277'},
    'components-acoustic-lambda-1': {"path": TRAINING_ROOT / 'components-acoustic-lambda-1' / '20261029T163838Z-22fbc98a', "manifest_sha256": '965310993ca218f3db9ffaa462fec06f833b2a952d42422aba0de3ec6e6f9162'},
    'components-acoustic-lambda-10': {"path": TRAINING_ROOT / 'components-acoustic-lambda-10' / '20261029T163912Z-380843a2', "manifest_sha256": '043fa7ec0ffa480ceeab44a208ea38d921de3092c6b8d814bb5cd662f91308d2'},
}
SUMMARY_RUN_ID = "training-summary-01"
PRESENTATION_ID = "training-figures-01"
# prepare reads training logs; reuse renders the exact existing summary without reading logs.
MODE = "prepare"
FONT_FILES = []
INSTALL_DEPENDENCIES = False
DEPENDENCIES = ["numpy", "matplotlib"]


## 2. Dependencies
NumPy and Matplotlib are the only nonstandard runtime dependencies.


In [13]:
from __future__ import annotations

import hashlib
import copy
import importlib.metadata
import json
import math
import os
import platform
import re
import shutil
import subprocess
import sys
import tempfile
from contextlib import contextmanager
from dataclasses import dataclass
from hashlib import sha256
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
from typing import Any, Iterable
from numbers import Real
from statistics import NormalDist

if sys.version_info < (3, 11):
    raise RuntimeError("Python 3.11 or newer is required")
import tomllib

if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *DEPENDENCIES])

_plot_cache = tempfile.TemporaryDirectory(prefix="evaluation-matplotlib-")
os.environ["MPLCONFIGDIR"] = _plot_cache.name
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator

print("numpy", np.__version__)
print("matplotlib", matplotlib.__version__)

from matplotlib.colors import to_rgb


numpy 2.2.6
matplotlib 3.10.1


In [14]:
BUILD_METADATA = {
    "schema_version": 1,
    "scope": "training_visualization",
    "source_sha256": {
        "research/artifacts.py": "663ec77c515bcbe1f350aa12b2d4fea80491ef10512069be387845f8c522c239",
        "evaluation/evaluation_outputs.py": "7c7c5fce3bba3026d967abb91b0325295ed88494add2337c827ac74740ffaa5f",
        "evaluation/training_summary.py": "ce3ac24cd6168faba0d32415b701305b9334438be95ba850f20d30a48d892938",
        "visualization/core.py": "ab70b4b4ff92482498c863ed090470f10e8bb38f24482b794ab4a1f1d9a5c325",
        "visualization/training.py": "90bb4281a57ef51ac30ef054a1cecbccfdf7a02405459be278da8767303db365"
    },
    "definition_sha256": {
        "research/artifacts.py": "4d17b9f045852b756ea23a2312aa2de35cf712d004151819fe852710e3538f86",
        "evaluation/evaluation_outputs.py": "9d305de299cbf62399447864f778cdfd7e18dfe8678c755c2b194a40923c3db8",
        "evaluation/training_summary.py": "d5252a043ddeb97fc4b16bd5955f4f043990526f032a70b1ecc2df340488bed1",
        "visualization/core.py": "0f779bfbf0459226ecbe068f6e4cac3c66d5f55d5497c41fb1a018723d50186b",
        "visualization/training.py": "e15dd0a0126706ae508b10cc69bd26dcb57b68dd80237cc019fc3406db327620"
    },
    "resource_sha256": {
        "visualization/style.toml": "5dcaf9f879cbe9629cc2670f7d6a5f1ab80f3271a032624274fa05210043f677",
        "visualization/registry.toml": "9872b154498a31eb0fd545cd62127d3f01ebd98ebb8e5269be91f674f51378e3"
    },
    "generator_sha256": "bfe49fb61847bba7d5f370795a566c523aa62a9ca4cc3c00c5d3cea34622acd4",
    "embedding_builder_sha256": "64f35b19a36808fb535f4c03f3eed069ea0678bb1a81170f7fffcd60ff6e6bbd"
}


## 3. Arial and shared style
Provide existing Arial files if the font is unavailable. No font downloads or substitution.


In [15]:
for filename in FONT_FILES:
    font_manager.fontManager.addfont(str(Path(filename).resolve(strict=True)))
if not any(font.name == "Arial" for font in font_manager.fontManager.ttflist):
    raise RuntimeError("Arial is unavailable; provide regular and bold FONT_FILES")
_style_resources = tempfile.TemporaryDirectory(prefix="training-style-")
STYLE_RESOURCE_ROOT = Path(_style_resources.name)


In [16]:
_ = (STYLE_RESOURCE_ROOT / 'style.toml').write_text('[schema]\nversion = 1\n\n[font]\nfamily = "Arial"\naxis_label_pt = 9.0\ntick_pt = 8.0\nlegend_pt = 8.0\npanel_heading_pt = 9.0\npanel_letter_pt = 10.0\nannotation_pt = 8.0\nnormal_weight = "normal"\npanel_letter_weight = "bold"\n\n[color]\nbackground = "#FFFFFF"\nforeground = "#171717"\nsecondary = "#666666"\ntertiary = "#888888"\nquaternary = "#999999"\n\n[line]\ndata_pt = 1.2\naxes_pt = 0.7\nreference_pt = 0.7\ninterval_pt = 0.9\ncaps_pt = 3.0\n\n[marker]\nsize_pt = 4.0\nedge_pt = 0.8\nface = "white"\nmarkevery_target = 10\n\n[layout]\nsingle_width_mm = 90.0\nmulti_width_mm = 180.0\npanel_aspect = 1.3333333333333333\nleft_mm = 14.0\nright_mm = 5.0\nbottom_mm = 12.0\ntop_mm = 8.0\nhgap_mm = 14.0\nvgap_mm = 16.0\nlegend_row_mm = 8.0\nlegend_gap_mm = 3.0\ndefault_columns = [1, 2, 2, 2, 3, 3, 3]\n\n[axes]\ngrid = false\nminor_ticks = false\ntick_direction = "in"\nmajor_length_pt = 3.0\nmajor_width_pt = 0.7\nmajor_pad_pt = 3.0\nmajor_target = 5\npanel_label_x_mm = -3.0\npanel_label_y_mm = 2.0\n\n[legend]\nmax_per_row = 4\nhandle_length_em = 2.4\ncolumn_spacing_em = 1.2\nlocation = "upper center"\n\n[histogram]\ndensity = true\nfill = false\nbootstrap_bins = 40\n\n[export]\nformats = ["png", "eps"]\ndpi = 300\ntransparent = false\noverwrite = false\n\n[backend.rcparams]\n"pdf.fonttype" = 42\n"ps.fonttype" = 42\n"svg.fonttype" = "none"\n', encoding='utf-8')
_ = (STYLE_RESOURCE_ROOT / 'registry.toml').write_text('[schema]\nversion = 1\n\n[figures.pilot_ecdf]\nstatus = "approved"\nkind = "ecdf"\nname = "Cosine distributions"\nxlabel = "Cosine similarity"\nylabel = "ECDF"\nseries = ["base", "zh_cn"]\norder = ["base", "zh_cn"]\nscale = "linear"\nxlim = [-1.0, 1.0]\nylim = [0.0, 1.0]\n\n[figures.pilot_histogram]\nstatus = "approved"\nkind = "histogram"\nname = "Cosine density"\nxlabel = "Cosine similarity"\nylabel = "Density"\nseries = ["base", "zh_cn"]\norder = ["base", "zh_cn"]\nscale = "linear"\ndensity = true\nxlim = [-1.0, 1.0]\nylim = [0.0]\nedges_start = -1.0\nedges_stop = 1.0\nedges_count = 61\n\n[figures.pilot_bootstrap]\nstatus = "approved"\nkind = "bootstrap_histogram"\nname = "Bootstrap differences"\nxlabel = "Zh-cn \\u2212 Base"\nylabel = "Bootstrap repeats"\nseries = ["delta"]\norder = ["delta"]\nscale = "linear"\nbins = 40\ndensity = false\nylim = [0.0]\n\n[semantics.base]\nlabel = "Base"\ncolor = "#171717"\nlinestyle = "solid"\nmarker = "o"\n\n[semantics.zh_cn]\nlabel = "Zh-cn"\ncolor = "#666666"\nlinestyle = "dashed"\nmarker = "s"\n\n[semantics.delta]\nlabel = "Zh-cn \\u2212 Base"\ncolor = "#171717"\nlinestyle = "solid"\nmarker = "o"\n\n[figures.dimsim_distributions]\nstatus = "approved"\nkind = "distribution_triptych"\nname = "DIMSIM distributions"\nxlabel = "DIMSIM distance"\nylabel = "Pair count / ECDF"\nseries = ["population"]\norder = ["population"]\nscale = "linear"\n\n[figures.dimsim_overlap_distributions]\nstatus = "approved"\nkind = "ecdf"\nname = "Component overlap distributions"\nxlabel = "DIMSIM distance"\nylabel = "ECDF"\nseries = ["000", "001", "010", "011", "100", "101", "110", "111"]\norder = ["000", "001", "010", "011", "100", "101", "110", "111"]\nscale = "linear"\nylim = [0.0, 1.0]\n\n[figures.dimsim_component_contributions]\nstatus = "approved"\nkind = "stacked_bar"\nname = "DIMSIM component contributions"\nxlabel = "DIMSIM interval"\nylabel = "Share of mean DIMSIM distance"\nseries = ["initial", "final", "tone"]\norder = ["initial", "final", "tone"]\nscale = "linear"\n\n[semantics.population]\nlabel = "All pairs"\ncolor = "#171717"\nlinestyle = "solid"\nmarker = "o"\n\n[semantics.000]\nlabel = "000"\ncolor = "#171717"\nlinestyle = "solid"\nmarker = "o"\n[semantics.001]\nlabel = "001"\ncolor = "#444444"\nlinestyle = "dashed"\nmarker = "s"\n[semantics.010]\nlabel = "010"\ncolor = "#666666"\nlinestyle = "dotted"\nmarker = "^"\n[semantics.011]\nlabel = "011"\ncolor = "#888888"\nlinestyle = "dashdot"\nmarker = "v"\n[semantics.100]\nlabel = "100"\ncolor = "#303030"\nlinestyle = "solid"\nmarker = "D"\n[semantics.101]\nlabel = "101"\ncolor = "#606060"\nlinestyle = "dashed"\nmarker = "P"\n[semantics.110]\nlabel = "110"\ncolor = "#909090"\nlinestyle = "dotted"\nmarker = "X"\n[semantics.111]\nlabel = "111"\ncolor = "#c0c0c0"\nlinestyle = "dashdot"\nmarker = "*"\n[semantics.initial]\nlabel = "Initial"\ncolor = "#303030"\nlinestyle = "solid"\nmarker = "o"\n[semantics.final]\nlabel = "Final"\ncolor = "#888888"\nlinestyle = "solid"\nmarker = "s"\n[semantics.tone]\nlabel = "Tone"\ncolor = "#d0d0d0"\nlinestyle = "solid"\nmarker = "^"\n\n[figures.anchor_q_transitions]\nstatus = "approved"\nkind = "dotplot"\nname = "Anchor-Q transitions"\nxlabel = "Q"\nylabel = "Transition"\nseries = ["syllable", "components", "acoustic", "components-acoustic-lambda-1", "components-acoustic-lambda-10"]\norder = ["syllable", "components", "acoustic", "components-acoustic-lambda-1", "components-acoustic-lambda-10"]\nscale = "linear"\n\n[figures.transition_shares]\nstatus = "approved"\nkind = "stacked_bar"\nname = "Transition shares"\nxlabel = "Transition"\nylabel = "Q share"\nseries = ["initial", "final", "tone"]\norder = ["initial", "final", "tone"]\nscale = "linear"\n\n[figures.component_shares]\nstatus = "approved"\nkind = "stacked_bar"\nname = "Component shares"\nxlabel = "Model"\nylabel = "Q share"\nseries = ["initial", "final", "tone"]\norder = ["initial", "final", "tone"]\nscale = "linear"\n\n[figures.dimsim_geometry]\nstatus = "approved"\nkind = "line_histogram"\nname = "DIMSIM geometry"\nxlabel = "DIMSIM distance"\nylabel = "Cosine similarity"\nseries = ["syllable", "components", "acoustic", "components-acoustic-lambda-1", "components-acoustic-lambda-10"]\norder = ["syllable", "components", "acoustic", "components-acoustic-lambda-1", "components-acoustic-lambda-10"]\nscale = "linear"\n\n[figures.dimsim_correlation]\nstatus = "approved"\nkind = "dotplot"\nname = "DIMSIM correlation"\nxlabel = "Model"\nylabel = "Spearman rho"\nseries = ["syllable", "components", "acoustic", "components-acoustic-lambda-1", "components-acoustic-lambda-10"]\norder = ["syllable", "components", "acoustic", "components-acoustic-lambda-1", "components-acoustic-lambda-10"]\nscale = "linear"\n\n[semantics.syllable]\nlabel = "Syllable"\ncolor = "#171717"\nlinestyle = "solid"\nmarker = "o"\n[semantics.components]\nlabel = "Components"\ncolor = "#444444"\nlinestyle = "dashed"\nmarker = "s"\n[semantics.acoustic]\nlabel = "Acoustic"\ncolor = "#707070"\nlinestyle = "dotted"\nmarker = "^"\n[semantics."components-acoustic-lambda-1"]\nlabel = "Combined \\u03bb1"\ncolor = "#999999"\nlinestyle = "dashdot"\nmarker = "D"\n[semantics."components-acoustic-lambda-10"]\nlabel = "Combined \\u03bb10"\ncolor = "#c5c5c5"\nlinestyle = "solid"\nmarker = "v"\n\n[figures.residual_qq]\nstatus = "approved"\nkind = "dotplot"\nname = "Residual Q\\u2013Q"\nxlabel = "Normal quantile"\nylabel = "Standardized residual"\nseries = ["syllable", "components", "acoustic", "components-acoustic-lambda-1", "components-acoustic-lambda-10", "dimsim"]\norder = ["syllable", "components", "acoustic", "components-acoustic-lambda-1", "components-acoustic-lambda-10", "dimsim"]\ncolumns = 2\nscale = "linear"\n\n[semantics.dimsim]\nlabel = "DIMSIM"\ncolor = "#171717"\nlinestyle = "solid"\nmarker = "o"\n\n[figures.training_objectives]\nstatus = "approved"\nkind = "line_histogram"\nname = "Training objectives"\nxlabel = "Epoch"\nylabel = "Objective"\nseries = ["syllable", "components", "acoustic", "components-acoustic-lambda-1", "components-acoustic-lambda-10"]\norder = ["syllable", "components", "acoustic", "components-acoustic-lambda-1", "components-acoustic-lambda-10"]\ncolumns = 3\nscale = "linear"\n\n[figures.validation_accuracy]\nstatus = "approved"\nkind = "line_histogram"\nname = "Validation accuracy"\nxlabel = "Epoch"\nylabel = "Accuracy"\nseries = ["syllable", "components", "components-acoustic-lambda-1", "components-acoustic-lambda-10"]\norder = ["syllable", "components", "components-acoustic-lambda-1", "components-acoustic-lambda-10"]\ncolumns = 2\nscale = "linear"\n\n[figures.validation_loss_components]\nstatus = "approved"\nkind = "line_histogram"\nname = "Validation loss components"\nxlabel = "Epoch"\nylabel = "Loss"\nseries = ["components", "acoustic", "components-acoustic-lambda-1", "components-acoustic-lambda-10"]\norder = ["components", "acoustic", "components-acoustic-lambda-1", "components-acoustic-lambda-10"]\ncolumns = 2\nscale = "linear"\n\n[figures.component_accuracy]\nstatus = "approved"\nkind = "line_histogram"\nname = "Component accuracy"\nxlabel = "Epoch"\nylabel = "Accuracy"\nseries = ["initial", "final", "tone"]\norder = ["initial", "final", "tone"]\ncolumns = 2\nscale = "linear"\n\n[figures.invalid_if_predictions]\nstatus = "approved"\nkind = "line_histogram"\nname = "Invalid initial\\u2013final combinations"\nxlabel = "Epoch"\nylabel = "Invalid predictions (%)"\nseries = ["components"]\norder = ["components"]\ncolumns = 1\nscale = "linear"\n', encoding='utf-8')


### Input and output paths


In [17]:
def file_sha256(path, chunk=1024 * 1024):
    h = sha256()
    with Path(path).open('rb') as f:
        while (data := f.read(chunk)):
            h.update(data)
    return h.hexdigest()

def safe_run_id(value):
    if not isinstance(value, str) or not re.fullmatch('[A-Za-z0-9][A-Za-z0-9._-]{0,127}', value) or value in {'.', '..'}:
        raise ValueError('run_id must be a non-empty safe filename component')
    return value

def safe_analysis(value):
    if not isinstance(value, str) or not re.fullmatch('[a-z0-9][a-z0-9-]{0,63}', value):
        raise ValueError('analysis must be a safe lowercase name')
    return value

def ensure_output_outside_inputs(output, input_roots):
    output = Path(output).resolve()
    for root in map(lambda p: Path(p).resolve(), input_roots):
        if output == root or root in output.parents:
            raise ValueError(f'Output must not be inside input: {root}')


### Verified numerical persistence


In [18]:
def data_json_bytes(value: Any) -> bytes:
    return (json.dumps(value, ensure_ascii=False, indent=2, sort_keys=True, allow_nan=False) + '\n').encode('utf-8')

def write_data_json(path: Path, value: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('xb') as stream:
        stream.write(data_json_bytes(value))

def artifact_files(root: Path) -> set[str]:
    return {p.relative_to(root).as_posix() for p in root.rglob('*') if p.is_file()}

def read_finite_json(path: Path):

    def bad(token):
        raise ValueError(f'non-finite JSON constant: {token}')
    value = json.loads(path.read_text(encoding='utf-8'), parse_constant=bad)
    data_json_bytes(value)
    return value

def check_input_pins(inputs, scientific_pins):
    bundle_pin = scientific_pins.get('bundle_manifest_sha256')
    if bundle_pin and bundle_pin not in {x['sha256'] for x in inputs}:
        raise ValueError('scientific bundle manifest pin does not match an input identity')
    exp_pin = scientific_pins.get('experiment_manifest_sha256')
    if exp_pin and exp_pin not in {x['sha256'] for x in inputs}:
        raise ValueError('scientific experiment manifest pin does not match an input identity')

def verify_artifact_inventory(root: Path, analysis: str, run_id: str, expected: set[str], *, profile: str='numerical_outputs', figure_exports: str='pending'):
    safe_analysis(analysis)
    safe_run_id(run_id)
    manifest = read_finite_json(root / 'manifest.json')
    if manifest.get('schema_version') != 1 or manifest.get('status') != 'complete' or manifest.get('analysis') != analysis or (manifest.get('run_id') != run_id) or (manifest.get('scope') != profile) or (manifest.get('figure_exports') != figure_exports) or (set(manifest.get('required_artifacts', [])) != expected):
        raise ValueError('output manifest identity/profile mismatch')
    if not isinstance(manifest.get('python'), str) or not manifest['python'] or (not isinstance(manifest.get('dependencies'), dict)) or ('numpy' not in manifest['dependencies']) or (not isinstance(manifest.get('inputs'), list)) or (not isinstance(manifest.get('source_hashes'), dict)) or (not manifest['source_hashes']):
        raise ValueError('output provenance metadata is incomplete')
    for row in manifest['inputs']:
        if not isinstance(row, dict) or not isinstance(row.get('path'), str) or (not isinstance(row.get('identity_path'), str)) or (not is_sha256(row.get('sha256'))):
            raise ValueError('output input identity metadata is invalid')
    if any((not is_sha256(x) for x in manifest['source_hashes'].values())):
        raise ValueError('output source hash metadata is invalid')
    actual, inventory = (artifact_files(root) - {'manifest.json'}, manifest.get('artifacts'))
    declared = {x.get('path') for x in inventory if isinstance(x, dict)} if isinstance(inventory, list) else set()
    if actual != expected or declared != expected or len(inventory or []) != len(expected):
        raise ValueError('output artifact inventory mismatch')
    for row in inventory:
        if not isinstance(row, dict) or set(row) != {'path', 'sha256', 'bytes'}:
            raise ValueError('malformed artifact inventory row')
        path = root / row['path']
        if not path.is_file() or path.stat().st_size <= 0 or path.stat().st_size != row['bytes'] or (file_sha256(path) != row['sha256']):
            raise ValueError(f"artifact hash or size mismatch: {row.get('path')}")
    return manifest

def is_sha256(value):
    return isinstance(value, str) and len(value) == 64 and all((c in '0123456789abcdef' for c in value))

@contextmanager
def presentation_run(project_root: str | Path, analysis: str, run_id: str, *, input_roots: list[str | Path], config: dict, source_hashes: dict, expected_files: set[str], scientific_pins: dict, create_data: bool=True, output_path: str | Path | None=None, profile: str='numerical_outputs', figure_exports_on_complete: str='pending'):
    """Write one run, gating every file except its top-level manifest."""
    analysis, run_id = (safe_analysis(analysis), safe_run_id(run_id))
    if figure_exports_on_complete not in {'pending', 'complete'}:
        raise ValueError('unsupported figure export completion status')
    root = Path(output_path).resolve() if output_path is not None else Path(project_root).resolve() / 'outputs' / analysis / run_id
    ensure_output_outside_inputs(root, input_roots)
    if root.exists():
        raise FileExistsError(f'refusing to overwrite existing output directory: {root}')
    if not isinstance(config, dict) or not isinstance(source_hashes, dict) or (not source_hashes):
        raise ValueError('explicit config and source hash map are required')
    if any((not isinstance(v, str) or len(v) != 64 or any((c not in '0123456789abcdef' for c in v)) for v in source_hashes.values())):
        raise ValueError('source hashes must be lowercase SHA-256 digests')
    data_json_bytes({'config': config, 'source_hashes': source_hashes, 'scientific_pins': scientific_pins})
    inputs = []
    for raw in input_roots:
        p = Path(raw).resolve(strict=True)
        identity = p if p.is_file() else p / 'manifest.json'
        if not identity.is_file():
            raise FileNotFoundError(f'input manifest identity missing: {identity}')
        inputs.append({'path': str(p), 'identity_path': str(identity), 'sha256': file_sha256(identity)})
    check_input_pins(inputs, scientific_pins)
    dependencies = {}
    for package in ('numpy', 'scipy', 'pandas', 'matplotlib'):
        try:
            dependencies[package] = version(package)
        except PackageNotFoundError:
            dependencies[package] = None
    root.mkdir(parents=True, exist_ok=False)
    manifest = {'schema_version': 1, 'analysis': analysis, 'run_id': run_id, 'status': 'running', 'scope': profile, 'figure_exports': 'pending', 'required_artifacts': sorted(expected_files), 'artifacts': [], 'scientific_input_pins': scientific_pins, 'source_hashes': source_hashes, 'inputs': inputs, 'python': platform.python_version(), 'dependencies': dependencies}
    write_data_json(root / 'manifest.json', manifest)
    try:
        names = (('data',) if create_data else ()) + ('tables', 'figures', 'diagnostics')
        for name in names:
            (root / name).mkdir()
        write_data_json(root / 'config.json', config)
        yield root
        actual = artifact_files(root) - {'manifest.json'}
        if actual != expected_files:
            raise RuntimeError(f'artifact inventory mismatch; missing={sorted(expected_files - actual)}, unexpected={sorted(actual - expected_files)}')
        artifacts = []
        for relative in sorted(actual):
            path = root / relative
            if path.stat().st_size <= 0:
                raise RuntimeError(f'required artifact is empty: {relative}')
            artifacts.append({'path': relative, 'sha256': file_sha256(path), 'bytes': path.stat().st_size})
        manifest.update(status='complete', artifacts=artifacts, figure_exports=figure_exports_on_complete)
    except BaseException as exc:
        manifest.update(status='failed', error={'type': type(exc).__name__, 'message': str(exc)})
        raise
    finally:
        fd, name = tempfile.mkstemp(prefix='manifest-', suffix='.tmp', dir=root)
        try:
            with os.fdopen(fd, 'wb') as stream:
                stream.write(data_json_bytes(manifest))
            os.replace(name, root / 'manifest.json')
        finally:
            if os.path.exists(name):
                os.unlink(name)


### Training logs and selected checkpoints


In [19]:
TRAINING_MODELS = ('syllable', 'components', 'acoustic', 'components-acoustic-lambda-1', 'components-acoustic-lambda-10')
TRAINING_SEEDS = (0, 1, 2, 3, 4)
METRICS_SOURCE_SHA256 = '10d41722970c5472cfe9c3f09346ae977d95dab520fc8d72a4f26d3e7f935fbc'
SELECTION_RULE = 'minimum per-seed validation objective; strict improvement keeps earliest tie'
TRAINING_CONFIG = {'seed_ids': list(TRAINING_SEEDS), 'selection': 'minimum validation objective; earliest exact tie', 'dispersion': 'sample SD across training seeds; not SE or CI', 'classification_component': 'mean of initial/final/tone CE', 'acoustic_component': 'unweighted 1 - cosine', 'invalid_if_scope': 'components validation only', 'bootstrap': False, 'test_used_for_selection': False, 'metrics_source_sha256': METRICS_SOURCE_SHA256}

def _training_number(value):
    return isinstance(value, (int, float)) and (not isinstance(value, bool)) and math.isfinite(value)

def _training_metrics(metrics, model, diagnostics, location):
    if not isinstance(metrics, dict) or not _training_number(metrics.get('objective')):
        raise ValueError(f'missing or invalid objective: {location}')
    if type(metrics.get('characters')) is not int or metrics['characters'] <= 0:
        raise ValueError(f'invalid character count: {location}')
    for key, value in metrics.items():
        if key.endswith(('accuracy', '_rate', '_ce')) or key in {'objective', 'cosine', 'weighted_cosine', 'component_ce'}:
            if not _training_number(value):
                raise ValueError(f'invalid metric {key}: {location}')
            if key.endswith(('accuracy', '_rate')) and (not 0 <= value <= 1):
                raise ValueError(f'rate outside [0,1]: {location}')

    def close(a, b):
        return math.isclose(a, b, rel_tol=1e-06, abs_tol=1e-08)
    components = ('initials_ce', 'finals_ce', 'tones_ce')
    if all((k in metrics for k in (*components, 'component_ce'))):
        if not close(metrics['component_ce'], sum((metrics[k] for k in components)) / 3):
            raise ValueError(f'component CE definition mismatch: {location}')
    if 'invalid_if_rate' in metrics and 'invalid_if_count' in metrics:
        count = metrics['invalid_if_count']
        if type(count) is not int or not 0 <= count <= metrics['characters'] or (not close(metrics['invalid_if_rate'], count / metrics['characters'])):
            raise ValueError(f'invalid-IF definition mismatch: {location}')
    terms = ['syllable_ce'] if model == 'syllable' else ['component_ce'] if model == 'components' else ['cosine'] if model == 'acoustic' else ['component_ce', 'cosine']
    if all((k in metrics for k in terms)):
        weight = 10.0 if model.endswith('lambda-10') else 1.0
        expected = metrics[terms[0]] if len(terms) == 1 else metrics['component_ce'] + weight * metrics['cosine']
        if not close(metrics['objective'], expected):
            raise ValueError(f'objective definition mismatch: {location}')
        if 'weighted_cosine' in metrics and 'cosine' in metrics and (not close(metrics['weighted_cosine'], weight * metrics['cosine'])):
            raise ValueError(f'weighted cosine definition mismatch: {location}')
    for key in terms:
        if key not in metrics:
            diagnostics.append({'location': location, 'metric': key, 'status': 'missing'})

def _training_stats(values):
    return {'seed_values': values, 'mean': float(np.mean(values)), 'sample_sd': float(np.std(values, ddof=1))}

def load_training_runs(training_runs):
    """Explicit model→{path,manifest_sha256}; only manifests/history/summary opened."""
    if not isinstance(training_runs, dict) or set(training_runs) != set(TRAINING_MODELS):
        raise ValueError('exact five training model paths and manifest pins required')
    result = {'config': dict(TRAINING_CONFIG), 'models': {}, 'selected': [], 'diagnostics': [], 'inputs': []}
    shared = None
    split_counts = {}
    selected_summary = []
    metrics_fields = {}
    for model in TRAINING_MODELS:
        entry = training_runs[model]
        root = Path(entry['path']).resolve(strict=True)
        sha = entry.get('manifest_sha256')
        if not is_sha256(sha) or file_sha256(root / 'manifest.json') != sha:
            raise ValueError(f'explicit training manifest hash mismatch: {model}')
        m = read_finite_json(root / 'manifest.json')
        if m.get('schema_version') != 1 or m.get('run_type') != 'training_run' or m.get('status') != 'complete' or (m.get('model_id') != model) or (m.get('run_id') != root.name) or (m.get('seeds') != list(TRAINING_SEEDS)) or any((type(s) is not int for s in m.get('seeds', []))) or (m.get('seed_status') != {str(s): 'complete' for s in TRAINING_SEEDS}) or (m.get('selection') != SELECTION_RULE) or (m.get('code_sha256', {}).get('metrics.py') != METRICS_SOURCE_SHA256):
            raise ValueError(f'incompatible training manifest/metric definitions: {model}')
        expected_weight = 10.0 if model.endswith('lambda-10') else 1.0 if model.endswith('lambda-1') else None
        if m.get('lambda_cosine') != expected_weight:
            raise ValueError('training lambda identity mismatch')
        signature = (m['experiment_id'], m['experiment_hashes'], m['config'])
        if shared is None:
            shared = signature
        if signature != shared:
            raise ValueError('training experiments/splits/configs differ')
        epochs = m['config']['training']['epochs']
        if type(epochs) is not int or epochs < 1:
            raise ValueError('invalid epoch count')
        if m['config']['training_seeds'] != list(TRAINING_SEEDS):
            raise ValueError('config seed identities differ')
        found = sorted((p.name for p in root.glob('seed-*') if p.is_dir()))
        if found != [f'seed-{s}' for s in TRAINING_SEEDS]:
            raise ValueError('seed directory inventory differs')
        files = {}

        def read_log(name):
            expected = m['artifacts'].get(name)
            path = root / name
            if not is_sha256(expected) or file_sha256(path) != expected:
                raise ValueError(f'training log hash mismatch: {model}/{name}')
            value = read_finite_json(path)
            files[name] = expected
            return value
        run_summary = read_log('summary.json')
        if run_summary.get('status') != 'complete' or run_summary.get('seeds_completed') != 5:
            raise ValueError('incomplete run summary')
        declared = run_summary.get('seed_results', [])
        by_seed = {row['seed']: row for row in declared}
        if len(declared) != 5 or set(by_seed) != set(TRAINING_SEEDS) or any((type(r['seed']) is not int for r in declared)):
            raise ValueError('run-summary seed inventory differs')
        histories = []
        selected = []
        for seed in TRAINING_SEEDS:
            history = read_log(f'seed-{seed}/history.json')['epochs']
            summary = read_log(f'seed-{seed}/summary.json')
            if [r['epoch'] for r in history] != list(range(1, epochs + 1)) or any((type(r['epoch']) is not int for r in history)):
                raise ValueError('epoch grids differ; pointwise five-seed summaries require aligned epochs')
            if summary.get('seed') != seed or type(summary.get('seed')) is not int or summary.get('status') != 'complete':
                raise ValueError('per-seed identity/status mismatch')
            if summary != by_seed[seed]:
                raise ValueError('per-seed and run summaries disagree')
            for row in history:
                for split in ('train', 'validation'):
                    values = row[split]
                    _training_metrics(values, model, result['diagnostics'], f"{model}/seed-{seed}/epoch-{row['epoch']}/{split}")
                    count = values['characters']
                    if split in split_counts and split_counts[split] != count:
                        raise ValueError('training split populations differ')
                    split_counts[split] = count
            best = min(history, key=lambda r: (r['validation']['objective'], r['epoch']))
            if type(summary.get('best_epoch')) is not int or best['epoch'] != summary['best_epoch'] or best['validation']['objective'] != summary.get('best_validation_objective') or (best['validation'] != summary.get('best_validation_metrics')):
                raise ValueError('selected checkpoint does not minimize validation objective with earliest tie')
            test = summary.get('test_metrics', {})
            if test:
                _training_metrics(test, model, result['diagnostics'], f'{model}/seed-{seed}/test')
                if 'test' in split_counts and split_counts['test'] != test['characters']:
                    raise ValueError('test populations differ')
                split_counts['test'] = test['characters']
            else:
                result['diagnostics'].append({'model': model, 'seed': seed, 'status': 'missing_test_metrics'})
            selected_row = {'model_id': model, 'seed': seed, 'best_epoch': best['epoch'], 'validation_objective': best['validation']['objective'], 'validation': best['validation'], 'test': test}
            result['selected'].append(selected_row)
            selected.append(selected_row)
            histories.append(history)
        curves = {}
        for split in ('train', 'validation'):
            names = sorted({k for h in histories for r in h for k in r[split] if k not in {'gradient_norms'}})
            curves[split] = {}
            for name in names:
                vectors = [[r[split].get(name) for r in h] for h in histories]
                if not all((_training_number(v) for vector in vectors for v in vector)):
                    result['diagnostics'].append({'model': model, 'split': split, 'metric': name, 'status': 'incomplete_five_seed_curve'})
                    continue
                array = np.asarray(vectors, dtype=float)
                curves[split][name] = {'mean': array.mean(axis=0).tolist(), 'sample_sd': array.std(axis=0, ddof=1).tolist(), 'seed_values': array.tolist()}
        result['models'][model] = {'epochs': list(range(1, epochs + 1)), 'curves': curves, 'selected': selected}
        fields = {}
        for split in ('validation', 'test'):
            names = sorted({k for r in selected for k in r[split] if k != 'gradient_norms'})
            fields[split] = {k: _training_stats([r[split][k] for r in selected]) for k in names if all((_training_number(r[split].get(k)) for r in selected))}
        fields['best_epoch'] = _training_stats([r['best_epoch'] for r in selected])
        selected_summary.append({'model_id': model, **fields})
        if file_sha256(root / 'manifest.json') != sha or any((file_sha256(root / name) != value for name, value in files.items())):
            raise ValueError('training logs changed during reading')
        result['inputs'].append({'model_id': model, 'path': str(root), 'manifest_sha256': sha, 'logs': files})
    result['selected_summary'] = selected_summary
    result['config']['split_counts'] = split_counts
    result['config']['experiment_id'] = shared[0]
    result['config']['experiment_hashes'] = shared[1]
    return result

def training_tables(data):

    def number(v):
        return '—' if v is None else f'{v:.6f}'
    header = '| Model | Seed | Best epoch | Validation objective | Validation accuracy | Test objective | Test accuracy |'
    lines = [header, '| --- | --- | --- | --- | --- | --- | --- |']
    detail = ['| Model | Seed | Split | Metric | Value |', '| --- | --- | --- | --- | --- |']
    for r in data['selected']:
        metric = 'syllable_accuracy' if r['model_id'] == 'syllable' else 'joint_ift_accuracy'
        lines.append('| ' + ' | '.join([r['model_id'], str(r['seed']), str(r['best_epoch']), number(r['validation_objective']), number(r['validation'].get(metric)), number(r['test'].get('objective')), number(r['test'].get(metric))]) + ' |')
        for split in ('validation', 'test'):
            for key, value in sorted(r[split].items()):
                if _training_number(value):
                    detail.append(f"| {r['model_id']} | {r['seed']} | {split} | {key} | {number(value)} |")
    aggregate = ['| Model | Split | Metric | Mean | Seed SD |', '| --- | --- | --- | --- | --- |']
    for row in data['selected_summary']:
        for split in ('validation', 'test'):
            for metric, stats in row[split].items():
                aggregate.append(f"| {row['model_id']} | {split} | {metric} | {number(stats['mean'])} | {number(stats['sample_sd'])} |")
        stat = row['best_epoch']
        aggregate.append(f"| {row['model_id']} | selection | best_epoch | {number(stat['mean'])} | {number(stat['sample_sd'])} |")
    return {'selected-checkpoints.md': '\n'.join(lines) + '\n', 'selected-metrics.md': '\n'.join(detail) + '\n', 'selected-summary.md': '\n'.join(aggregate) + '\n'}

def write_training_summary(project_root, run_id, data, *, source_hashes):
    tables = training_tables(data)
    expected = {'config.json', 'data/training_summary.json', 'diagnostics/summary.json', *('tables/' + k for k in tables)}
    with presentation_run(project_root, 'training-summary', run_id, input_roots=[r['path'] for r in data['inputs']], config=data['config'], source_hashes=source_hashes, expected_files=expected, scientific_pins={'training_manifests': {r['model_id']: r['manifest_sha256'] for r in data['inputs']}}) as root:
        write_data_json(root / 'data/training_summary.json', data)
        write_data_json(root / 'diagnostics/summary.json', {'diagnostics': data['diagnostics'], 'selected_count': len(data['selected'])})
        for name, body in tables.items():
            with (root / 'tables' / name).open('x', encoding='utf-8', newline='\n') as stream:
                stream.write(body)
    return root

def load_training_summary(root):
    root = Path(root).resolve(strict=True)
    expected = {'config.json', 'data/training_summary.json', 'diagnostics/summary.json', 'tables/selected-checkpoints.md', 'tables/selected-metrics.md', 'tables/selected-summary.md'}
    manifest = verify_artifact_inventory(root, 'training-summary', root.name, expected)
    data = read_finite_json(root / 'data/training_summary.json')
    if data['config'] != read_finite_json(root / 'config.json') or {r['model_id']: r['manifest_sha256'] for r in data['inputs']} != manifest['scientific_input_pins']['training_manifests']:
        raise ValueError('training summary input identities disagree')
    for name, body in training_tables(data).items():
        if (root / 'tables' / name).read_text(encoding='utf-8') != body:
            raise ValueError('training table view mismatch')
    return (data, manifest)


### Shared figure protocol


In [20]:
ROOT = STYLE_RESOURCE_ROOT
STYLE_PATH = ROOT / 'style.toml'
REGISTRY_PATH = ROOT / 'registry.toml'
MM_PER_INCH = 25.4

class VisualizationError(ValueError):
    """An actionable protocol violation."""
STYLE_KEYS = {'schema': {'version'}, 'font': {'family', 'axis_label_pt', 'tick_pt', 'legend_pt', 'panel_heading_pt', 'panel_letter_pt', 'annotation_pt', 'normal_weight', 'panel_letter_weight'}, 'color': {'background', 'foreground', 'secondary', 'tertiary', 'quaternary'}, 'line': {'data_pt', 'axes_pt', 'reference_pt', 'interval_pt', 'caps_pt'}, 'marker': {'size_pt', 'edge_pt', 'face', 'markevery_target'}, 'layout': {'single_width_mm', 'multi_width_mm', 'panel_aspect', 'left_mm', 'right_mm', 'bottom_mm', 'top_mm', 'hgap_mm', 'vgap_mm', 'legend_row_mm', 'legend_gap_mm', 'default_columns'}, 'axes': {'grid', 'minor_ticks', 'tick_direction', 'major_length_pt', 'major_width_pt', 'major_pad_pt', 'major_target', 'panel_label_x_mm', 'panel_label_y_mm'}, 'legend': {'max_per_row', 'handle_length_em', 'column_spacing_em', 'location'}, 'histogram': {'density', 'fill', 'bootstrap_bins'}, 'export': {'formats', 'dpi', 'transparent', 'overwrite'}, 'backend': {'rcparams'}}
FIGURE_KEYS = {'status', 'kind', 'name', 'xlabel', 'ylabel', 'series', 'order', 'scale', 'bin_rule', 'bins', 'density', 'columns', 'xlim', 'ylim', 'edges_start', 'edges_stop', 'edges_count'}
SEMANTIC_KEYS = {'label', 'color', 'linestyle', 'marker'}
ALLOWED_RCPARAMS = {'pdf.fonttype', 'ps.fonttype', 'svg.fonttype'}

def read_style_toml(p):
    raw = Path(p).read_bytes()
    return (tomllib.loads(raw.decode()), raw)

def reject_unknown_style_keys(d, a, w):
    e = set(d) - a
    if e:
        raise VisualizationError(f"Unknown key(s) in {w}: {', '.join(sorted(e))}")

def require_positive_style_value(v, n):
    if isinstance(v, bool) or not isinstance(v, (int, float)) or (not math.isfinite(v)) or (v <= 0):
        raise VisualizationError(f'{n} must be finite and positive')

@dataclass(frozen=True)
class Protocol:
    style: dict[str, Any]
    registry: dict[str, Any]
    style_hash: str
    registry_hash: str
    style_path: Path
    registry_path: Path

    def figure(self, fid):
        try:
            return self.registry['figures'][fid]
        except KeyError as e:
            raise VisualizationError(f'Unknown figure ID: {fid}') from e

def load_protocol(style_path=STYLE_PATH, registry_path=REGISTRY_PATH):
    sp = Path(style_path).resolve()
    rp = Path(registry_path).resolve()
    s, sr = read_style_toml(sp)
    r, rr = read_style_toml(rp)
    reject_unknown_style_keys(s, set(STYLE_KEYS), 'style root')
    for sec, keys in STYLE_KEYS.items():
        if sec not in s:
            raise VisualizationError(f'Missing style section: {sec}')
        reject_unknown_style_keys(s[sec], keys, f'style.{sec}')
        missing = keys - set(s[sec])
        if missing:
            raise VisualizationError(f"Missing key(s) in style.{sec}: {', '.join(sorted(missing))}")
    if s['schema']['version'] != 1:
        raise VisualizationError('Unsupported style schema version')
    if s['font']['family'] != 'Arial':
        raise VisualizationError('The protocol requires Arial')
    for k, v in s['font'].items():
        if k.endswith('_pt'):
            require_positive_style_value(v, f'font.{k}')
    if (s['font']['normal_weight'], s['font']['panel_letter_weight']) != ('normal', 'bold'):
        raise VisualizationError('Font weights must be normal and bold')
    for k, v in s['layout'].items():
        if k != 'default_columns':
            require_positive_style_value(v, f'layout.{k}')
    if s['layout']['default_columns'] != [1, 2, 2, 2, 3, 3, 3]:
        raise VisualizationError('Invalid default columns')
    a = s['axes']
    if a['grid'] is not False or a['minor_ticks'] is not False or a['tick_direction'] != 'in':
        raise VisualizationError('Axes require grid/minor false and inward ticks')
    for k in ('major_length_pt', 'major_width_pt', 'major_pad_pt', 'major_target'):
        require_positive_style_value(a[k], f'axes.{k}')
    if s['export']['formats'] != ['png', 'eps'] or s['export']['dpi'] != 300 or s['export']['transparent'] is not False:
        raise VisualizationError('Export requires opaque PNG300 then EPS')
    reject_unknown_style_keys(s['backend']['rcparams'], ALLOWED_RCPARAMS, 'style.backend.rcparams')
    reject_unknown_style_keys(r, {'schema', 'figures', 'semantics'}, 'registry root')
    if r.get('schema', {}).get('version') != 1 or not isinstance(r.get('figures'), dict) or (not isinstance(r.get('semantics'), dict)):
        raise VisualizationError('Invalid registry schema')
    for sid, sem in r['semantics'].items():
        reject_unknown_style_keys(sem, SEMANTIC_KEYS, f'semantics.{sid}')
        if set(sem) != SEMANTIC_KEYS or not all((isinstance(x, str) and x.strip() for x in sem.values())):
            raise VisualizationError(f'Invalid semantics.{sid}')
    for fid, f in r['figures'].items():
        reject_unknown_style_keys(f, FIGURE_KEYS, f'figures.{fid}')
        req = {'status', 'kind', 'name', 'xlabel', 'ylabel', 'series', 'order', 'scale'}
        if req - set(f):
            raise VisualizationError(f'Missing required registry keys for {fid}')
        if f['status'] not in {'draft', 'approved'} or f['kind'] not in {'ecdf', 'histogram', 'bootstrap_histogram', 'distribution_triptych', 'stacked_bar', 'dotplot', 'line_histogram'} or f['scale'] not in {'linear', 'log'}:
            raise VisualizationError(f'Invalid status/kind/scale for {fid}')
        if not all((isinstance(f[k], str) and f[k].strip() for k in ('name', 'xlabel', 'ylabel'))):
            raise VisualizationError(f'Invalid labels for {fid}')
        if f['series'] != f['order'] or not f['series'] or len(set(f['series'])) != len(f['series']) or set(f['series']) - set(r['semantics']):
            raise VisualizationError(f'Invalid series/order for {fid}')
        if 'bin_rule' in f and f['bin_rule'] != 'pooled_fd_sturges':
            raise VisualizationError(f'Invalid bin_rule for {fid}')
        if 'bins' in f and (isinstance(f['bins'], bool) or not isinstance(f['bins'], int) or f['bins'] < 1):
            raise VisualizationError(f'Invalid bins for {fid}')
        if 'xlim' in f and (not isinstance(f['xlim'], list) or len(f['xlim']) != 2 or (not all((isinstance(x, (int, float)) and (not isinstance(x, bool)) and math.isfinite(x) for x in f['xlim']))) or (f['xlim'][0] >= f['xlim'][1])):
            raise VisualizationError(f'Invalid xlim for {fid}')
        if f['scale'] == 'log' and 'xlim' in f and (f['xlim'][0] <= 0):
            raise VisualizationError(f'Log xlim must be positive for {fid}')
        if f['kind'] == 'ecdf':
            if 'ylim' not in f or not isinstance(f['ylim'], list) or len(f['ylim']) != 2 or (not all((isinstance(x, (int, float)) and (not isinstance(x, bool)) and math.isfinite(x) for x in f['ylim']))) or (f['ylim'][0] >= f['ylim'][1]):
                raise VisualizationError(f'ECDF ylim must be two ascending finite values for {fid}')
        elif f['kind'] in {'histogram', 'bootstrap_histogram'}:
            if not isinstance(f.get('density'), bool):
                raise VisualizationError(f'Histogram density must be boolean for {fid}')
            y = f.get('ylim')
            if not isinstance(y, list) or len(y) not in {1, 2} or (not all((isinstance(x, (int, float)) and (not isinstance(x, bool)) and math.isfinite(x) for x in y))) or (len(y) == 2 and y[0] >= y[1]):
                raise VisualizationError(f'Histogram ylim must be a finite bottom or ascending pair for {fid}')
            if f['kind'] == 'histogram' and (not ({'edges_start', 'edges_stop', 'edges_count'} <= set(f) or f.get('bin_rule') == 'pooled_fd_sturges')):
                raise VisualizationError(f'Histogram requires fixed edges or pooled bin rule for {fid}')
            if f['kind'] == 'bootstrap_histogram' and 'bins' not in f:
                raise VisualizationError(f'Bootstrap histogram requires bins for {fid}')
        edgekeys = {'edges_start', 'edges_stop', 'edges_count'} & set(f)
        if edgekeys and (edgekeys != {'edges_start', 'edges_stop', 'edges_count'} or not all((isinstance(f[k], (int, float)) and (not isinstance(f[k], bool)) and math.isfinite(f[k]) for k in ('edges_start', 'edges_stop'))) or f['edges_stop'] <= f['edges_start'] or isinstance(f['edges_count'], bool) or (not isinstance(f['edges_count'], int)) or (f['edges_count'] < 2)):
            raise VisualizationError(f'Invalid edge specification for {fid}')
    return Protocol(s, r, sha256(sr).hexdigest(), sha256(rr).hexdigest(), sp, rp)

def verify_style_integrity(p):
    if sha256(p.style_path.read_bytes()).hexdigest() != p.style_hash or sha256(p.registry_path.read_bytes()).hexdigest() != p.registry_hash:
        raise VisualizationError('Protocol files changed; reload')
    if tomllib.loads(p.style_path.read_text(encoding='utf-8')) != p.style or tomllib.loads(p.registry_path.read_text(encoding='utf-8')) != p.registry:
        raise VisualizationError('Protocol mappings were mutated; reload')

def require_arial():
    from matplotlib import font_manager
    if not any((f.name == 'Arial' for f in font_manager.fontManager.ttflist)):
        raise VisualizationError('Arial is not installed; font fallback is prohibited')

def apply_style(p):
    import matplotlib as mpl
    require_arial()
    s = p.style
    f = s['font']
    a = s['axes']
    c = s['color']
    rc = {'font.family': f['family'], 'font.weight': f['normal_weight'], 'font.size': f['tick_pt'], 'axes.labelsize': f['axis_label_pt'], 'axes.titlesize': f['panel_heading_pt'], 'axes.labelweight': f['normal_weight'], 'axes.titleweight': f['normal_weight'], 'xtick.labelsize': f['tick_pt'], 'ytick.labelsize': f['tick_pt'], 'legend.fontsize': f['legend_pt'], 'axes.linewidth': s['line']['axes_pt'], 'lines.linewidth': s['line']['data_pt'], 'lines.markersize': s['marker']['size_pt'], 'lines.markeredgewidth': s['marker']['edge_pt'], 'lines.markerfacecolor': s['marker']['face'], 'axes.edgecolor': c['foreground'], 'axes.labelcolor': c['foreground'], 'text.color': c['foreground'], 'xtick.color': c['foreground'], 'ytick.color': c['foreground'], 'figure.facecolor': c['background'], 'axes.facecolor': c['background'], 'axes.grid': a['grid'], 'xtick.direction': a['tick_direction'], 'ytick.direction': a['tick_direction'], 'xtick.top': False, 'ytick.right': False, 'xtick.minor.visible': False, 'ytick.minor.visible': False, 'xtick.major.size': a['major_length_pt'], 'ytick.major.size': a['major_length_pt'], 'xtick.major.width': a['major_width_pt'], 'ytick.major.width': a['major_width_pt'], 'xtick.major.pad': a['major_pad_pt'], 'ytick.major.pad': a['major_pad_pt'], 'axes.spines.left': True, 'axes.spines.right': True, 'axes.spines.top': True, 'axes.spines.bottom': True}
    rc.update(s['backend']['rcparams'])
    mpl.rcParams.update(rc)

def layout(n, p, *, columns=None, legend_rows=0, width_mm=None):
    import matplotlib.pyplot as plt
    from matplotlib.ticker import MaxNLocator
    if not isinstance(n, int) or not 1 <= n <= 26:
        raise VisualizationError('Panel count must be 1..26')
    l = p.style['layout']
    cols = columns if columns is not None else l['default_columns'][n - 1] if n <= 7 else 3
    if not isinstance(cols, int) or not 1 <= cols <= min(3, n):
        raise VisualizationError('columns must be 1..min(3,n)')
    if width_mm is not None:
        require_positive_style_value(width_mm, 'width_mm')
        width = float(width_mm)
    else:
        width = l['single_width_mm'] if cols == 1 else l['multi_width_mm']
    rows = (n + cols - 1) // cols
    pw = (width - l['left_mm'] - l['right_mm'] - (cols - 1) * l['hgap_mm']) / cols
    if pw <= 0:
        raise VisualizationError('width_mm is too small for physical margins')
    ph = pw / l['panel_aspect']
    leg = legend_rows * l['legend_row_mm'] + (l['legend_gap_mm'] if legend_rows else 0)
    height = l['bottom_mm'] + l['top_mm'] + leg + rows * ph + (rows - 1) * l['vgap_mm']
    fig = plt.figure(figsize=(width / MM_PER_INCH, height / MM_PER_INCH))
    axes = []
    a = p.style['axes']
    for i in range(n):
        row = i // cols
        nr = min(cols, n - row * cols)
        col = i % cols
        x = l['left_mm'] + (cols - nr) * (pw + l['hgap_mm']) / 2 + col * (pw + l['hgap_mm'])
        y = l['bottom_mm'] + (rows - 1 - row) * (ph + l['vgap_mm'])
        ax = fig.add_axes([x / width, y / height, pw / width, ph / height])
        axes.append(ax)
        ax.tick_params(top=False, right=False, bottom=True, left=True, direction='in', length=a['major_length_pt'], width=a['major_width_pt'], pad=a['major_pad_pt'])
        ax.minorticks_off()
        ax.grid(False)
        ax.xaxis.set_major_locator(MaxNLocator(a['major_target']))
        ax.yaxis.set_major_locator(MaxNLocator(a['major_target']))
        ax.text(a['panel_label_x_mm'] / pw, 1 + a['panel_label_y_mm'] / ph, chr(97 + i), transform=ax.transAxes, fontsize=p.style['font']['panel_letter_pt'], fontweight=p.style['font']['panel_letter_weight'], va='bottom')
    return (fig, axes)

def finite(values, *, policy='raise'):
    if policy not in {'raise', 'drop'}:
        raise VisualizationError('finite_policy must be raise or drop')
    a = np.asarray(values, float)
    m = np.isfinite(a)
    if not m.all() and policy == 'raise':
        raise VisualizationError('Non-finite values present; select drop explicitly')
    a = a[m]
    if not len(a):
        raise VisualizationError('No finite observations')
    return a

def ecdf(values, *, finite_policy='raise'):
    x = np.sort(finite(values, policy=finite_policy), kind='mergesort')
    return (x, np.arange(1, len(x) + 1) / len(x))

def pooled_edges(series, *, finite_policy='raise'):
    x = np.concatenate([finite(v, policy=finite_policy) for v in series])
    if x.min() == x.max():
        return np.array([x[0] - 0.5, x[0] + 0.5])
    q = np.percentile(x, [25, 75])
    iqr = q[1] - q[0]
    bins = int(np.ceil((x.max() - x.min()) / (2 * iqr / np.cbrt(len(x))))) if iqr > 0 else int(np.ceil(np.log2(len(x))) + 1)
    return np.linspace(x.min(), x.max(), max(1, bins) + 1)

def _record(fig, d):
    fig._phonetics_resolution = d

def plot_ecdf(ax, series, figure, p, *, finite_policy='raise'):
    counts = {}
    for sid in figure['order']:
        x, y = ecdf(series[sid], finite_policy=finite_policy)
        counts[sid] = len(x)
        sem = p.registry['semantics'][sid]
        ax.step(x, y, where='post', label=sem['label'], color=sem['color'], linestyle=sem['linestyle'], marker=None, linewidth=p.style['line']['data_pt'])
    ax.set(xlabel=figure['xlabel'], ylabel=figure['ylabel'], ylim=tuple(figure['ylim']), xscale=figure['scale'])
    ax.set_xlim(tuple(figure['xlim'])) if 'xlim' in figure else None
    _record(ax.figure, {'kind': 'ecdf', 'finite_policy': finite_policy, 'counts': counts, 'limits': {'x': list(ax.get_xlim()), 'y': list(ax.get_ylim())}})

def plot_histogram(ax, series, figure, p, *, edges=None, finite_policy='raise'):
    arr = [finite(series[s], policy=finite_policy) for s in figure['order']]
    edges = np.linspace(figure['edges_start'], figure['edges_stop'], figure['edges_count']) if edges is None and 'edges_count' in figure else edges
    edges = np.histogram_bin_edges(np.concatenate(arr), bins=figure['bins']) if edges is None and 'bins' in figure else pooled_edges(arr) if edges is None else np.asarray(edges, float)
    edges = np.asarray(edges, float)
    if len(edges) < 2 or not np.all(np.isfinite(edges)) or (not np.all(np.diff(edges) > 0)):
        raise VisualizationError('Histogram edges must be finite and strictly increasing')
    for sid, x in zip(figure['order'], arr):
        sem = p.registry['semantics'][sid]
        ax.hist(x, bins=edges, density=figure['density'], histtype='step', label=sem['label'], color=sem['color'], linestyle=sem['linestyle'], linewidth=p.style['line']['data_pt'])
    ax.set(xlabel=figure['xlabel'], ylabel=figure['ylabel'], xscale=figure['scale'])
    ax.set_ylim(tuple(figure['ylim'])) if len(figure['ylim']) == 2 else ax.set_ylim(bottom=figure['ylim'][0])
    ax.set_xlim(tuple(figure['xlim'])) if 'xlim' in figure else None
    _record(ax.figure, {'kind': figure['kind'], 'histogram_edges': edges.tolist(), 'finite_policy': finite_policy, 'counts': {s: len(x) for s, x in zip(figure['order'], arr)}, 'limits': {'x': list(ax.get_xlim()), 'y': list(ax.get_ylim())}})
    return edges

def shared_legend(fig, axes, p, *, rows=1):
    h, l = axes[0].get_legend_handles_labels()
    c = p.style['legend']
    return fig.legend(h, l, loc=c['location'], ncol=min(c['max_per_row'], max(1, int(np.ceil(len(l) / rows)))), frameon=False, handlelength=c['handle_length_em'], columnspacing=c['column_spacing_em'])

def export_figure(fig, output_dir, stem, figure_id, p, *, scientific_provenance, overwrite=None):
    verify_style_integrity(p)
    spec = p.figure(figure_id)
    if spec['status'] != 'approved':
        raise VisualizationError(f'Figure {figure_id!r} is draft; approval required')
    if not re.fullmatch('[A-Za-z0-9][A-Za-z0-9._-]*', stem) or stem in {'.', '..'}:
        raise VisualizationError('Unsafe export stem')
    prov = dict(scientific_provenance)
    if not re.fullmatch('[0-9a-fA-F]{64}', str(prov.get('source_digest', ''))):
        raise VisualizationError('source_digest must be SHA-256 hex')
    require_arial()
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    if not out.is_dir():
        raise VisualizationError('Export target is not a directory')
    paths = [out / f'{stem}.{x}' for x in p.style['export']['formats']]
    mp = out / f'{stem}.metadata.json'
    allow = p.style['export']['overwrite'] if overwrite is None else overwrite
    if not allow and any((x.exists() for x in paths + [mp])):
        raise FileExistsError('Refusing to overwrite export or metadata')
    from matplotlib.colors import to_rgba_array
    for a in fig.findobj():
        if hasattr(a, 'get_visible') and (not a.get_visible()):
            continue
        alpha = a.get_alpha() if hasattr(a, 'get_alpha') else None
        if alpha is not None and 0 < alpha < 1:
            raise VisualizationError('EPS forbids partial transparency')
        for getter in ('get_color', 'get_markerfacecolor', 'get_markeredgecolor', 'get_facecolor', 'get_edgecolor'):
            if not hasattr(a, getter):
                continue
            try:
                rgba = to_rgba_array(getattr(a, getter)())
            except (TypeError, ValueError):
                continue
            if rgba.size and np.any((rgba[:, 3] > 0) & (rgba[:, 3] < 1)):
                raise VisualizationError('EPS forbids RGBA partial transparency')
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    box = fig.get_window_extent(renderer)
    from matplotlib.legend import Legend
    from matplotlib.text import Text
    ticks = set()
    for ax in fig.axes:
        for axis in (ax.xaxis, ax.yaxis):
            lo, hi = sorted(axis.get_view_interval())
            tol = max(1, abs(hi - lo)) * 1e-12
            for tick in axis._update_ticks():
                loc = tick.get_loc()
                if math.isfinite(loc) and lo - tol <= loc <= hi + tol:
                    ticks.update((x for x in (tick.label1, tick.label2) if x.get_visible()))
    for a in fig.findobj(lambda x: isinstance(x, (Text, Legend))):
        if isinstance(a, Text) and a not in ticks and any((a is z for ax in fig.axes for z in ax.get_xticklabels() + ax.get_yticklabels())):
            continue
        if not a.get_visible():
            continue
        b = a.get_window_extent(renderer)
        if b.width and b.height and (b.x0 < box.x0 - 1 or b.y0 < box.y0 - 1 or b.x1 > box.x1 + 1 or (b.y1 > box.y1 + 1)):
            raise VisualizationError('Visible text or legend is clipped')
    meta = {'schema_version': 1, 'config_schema_version': p.style['schema']['version'], 'figure_id': figure_id, 'style_hash': p.style_hash, 'registry_hash': p.registry_hash, 'resolved_style': p.style, 'resolved_figure': spec, 'resolved_data': getattr(fig, '_phonetics_resolution', {}), 'scientific_provenance': prov}
    temps = []
    try:
        for dest in paths:
            t = dest.with_name(dest.stem + '.tmp' + dest.suffix)
            temps.append(t)
            import matplotlib as mpl
            with mpl.rc_context({'savefig.bbox': None, 'savefig.pad_inches': 0, 'savefig.dpi': p.style['export']['dpi'], 'savefig.facecolor': p.style['color']['background'], 'savefig.transparent': False}):
                fig.savefig(t, format=dest.suffix[1:], dpi=p.style['export']['dpi'], bbox_inches=None, pad_inches=0, transparent=False, facecolor=p.style['color']['background'], metadata={'Creator': 'phonetics.visualization'})
        mt = mp.with_name(mp.stem + '.tmp' + mp.suffix)
        temps.append(mt)
        mt.write_text(json.dumps(meta, indent=2, sort_keys=True))
        for t, d in zip(temps, paths + [mp]):
            os.replace(t, d)
    finally:
        for t in temps:
            if t.exists():
                t.unlink()
    return paths


### Training figures and report


In [21]:
TRAINING_FIGURES = ('training_objectives', 'validation_accuracy', 'validation_loss_components', 'component_accuracy', 'invalid_if_predictions')
TRAINING_STEMS = ('training-objectives', 'validation-accuracy', 'validation-loss-components', 'component-accuracy', 'invalid-initial-final-combinations')

def _training_shade(color):
    return tuple((0.15 * v + 0.85 for v in to_rgb(color)))

def _training_curve(ax, data, model, metric, p, *, split='validation', semantic=None, scale=1.0, band=True, linestyle=None):
    item = data['models'][model]
    curve = item['curves'][split].get(metric)
    if curve is None:
        missing = getattr(ax, '_training_missing', []) + [f'{model}: {metric}']
        ax._training_missing = missing
        return
    sem = semantic or p.registry['semantics'][model]
    x = item['epochs']
    mean = np.asarray(curve['mean']) * scale
    sd = np.asarray(curve['sample_sd']) * scale
    if band:
        ax.fill_between(x, mean - sd, mean + sd, facecolor=_training_shade(sem['color']), edgecolor='none', zorder=0)
    ax.plot(x, mean, color=sem['color'], ls=linestyle or sem['linestyle'], marker=sem['marker'], mfc='white', markevery=max(1, len(x) // 10), lw=p.style['line']['data_pt'], ms=p.style['marker']['size_pt'], zorder=2)

def _training_legend(fig, handles, p, rows=1):
    fig.legend(handles=handles, loc=p.style['legend']['location'], frameon=False, ncol=min(p.style['legend']['max_per_row'], int(np.ceil(len(handles) / rows))), handlelength=p.style['legend']['handle_length_em'], columnspacing=p.style['legend']['column_spacing_em'])

def _training_model_handle(model, p):
    s = p.registry['semantics'][model]
    return Line2D([], [], color=s['color'], ls=s['linestyle'], marker=s['marker'], mfc='white', label=s['label'], ms=p.style['marker']['size_pt'])

def _training_finish(fig, axes, data, p, fid, *, shared_y=False):
    n = len(data['models'][TRAINING_MODELS[0]]['epochs'])
    for ax in axes:
        ax.set_xlabel('Epoch')
        ax.set_xlim((1, n) if n > 1 else (0.5, 1.5))
        ax.xaxis.set_major_locator(MaxNLocator(nbins=p.style['axes']['major_target'], integer=True))
        missing = getattr(ax, '_training_missing', [])
        if missing:
            ax.text(0.5, 0.5, 'Unavailable metric', transform=ax.transAxes, ha='center', fontsize=p.style['font']['annotation_pt'])
    if shared_y:
        limits = [ax.get_ylim() for ax in axes if ax.lines]
        if limits:
            lo = min((v[0] for v in limits))
            hi = max((v[1] for v in limits))
            for ax in axes:
                ax.set_ylim(lo, hi)
    fig._phonetics_resolution = {'kind': fid, 'dispersion': 'pointwise sample SD across five training seeds', 'seed_ids': [0, 1, 2, 3, 4], 'confidence_interval': False, 'band_encoding': 'opaque preblended gray for EPS', 'missing_metrics': [m for ax in axes for m in getattr(ax, '_training_missing', [])], 'axis_limits': [{'x': list(ax.get_xlim()), 'y': list(ax.get_ylim())} for ax in axes]}

def build_training_figures(data, protocol=None):
    p = protocol or load_protocol()
    figures = []
    sd_patch = Patch(facecolor=_training_shade('#171717'), label='± SD across seeds')
    try:
        fig, axes = layout(5, p, columns=3, legend_rows=1, width_mm=180)
        figures.append(fig)
        for ax, model in zip(axes, TRAINING_MODELS):
            semantic = {'color': p.registry['semantics'][model]['color'], 'linestyle': 'solid', 'marker': 'None'}
            _training_curve(ax, data, model, 'objective', p, semantic=semantic, linestyle='solid')
            _training_curve(ax, data, model, 'objective', p, split='train', semantic=semantic, band=False, linestyle='dashed')
            rows = data['models'][model]['selected']
            ax.plot([r['best_epoch'] for r in rows], [r['validation_objective'] for r in rows], ls='none', marker='o', mfc='white', mec=semantic['color'], ms=p.style['marker']['size_pt'], zorder=3)
            ax.set(title=p.registry['semantics'][model]['label'], ylabel='Objective')
        _training_legend(fig, [Line2D([], [], color='#171717', label='Validation'), Line2D([], [], color='#171717', ls='--', label='Training'), Patch(facecolor=_training_shade('#171717'), label='Validation ± SD'), Line2D([], [], color='#171717', ls='none', marker='o', mfc='white', label='Selected epoch')], p)
        _training_finish(fig, axes, data, p, 'training_objectives')
        fig, axes = layout(4, p, columns=2, legend_rows=1, width_mm=180)
        figures.append(fig)
        models = [m for m in TRAINING_MODELS if m != 'acoustic']
        for ax, model in zip(axes, models):
            _training_curve(ax, data, model, 'syllable_accuracy' if model == 'syllable' else 'joint_ift_accuracy', p)
            ax.set(title=p.registry['semantics'][model]['label'], ylabel='Accuracy')
        _training_legend(fig, [Line2D([], [], color='#171717', label='Validation'), sd_patch], p)
        _training_finish(fig, axes, data, p, 'validation_accuracy', shared_y=True)
        fig, axes = layout(2, p, columns=2, legend_rows=2, width_mm=180)
        figures.append(fig)
        for ax, models, metric, title in [(axes[0], ['components', *TRAINING_MODELS[3:]], 'component_ce', 'Component classification'), (axes[1], ['acoustic', *TRAINING_MODELS[3:]], 'cosine', 'Acoustic alignment')]:
            for model in models:
                _training_curve(ax, data, model, metric, p)
            ax.set(title=title, ylabel='Mean component CE' if metric == 'component_ce' else 'Unweighted cosine loss')
        _training_legend(fig, [_training_model_handle(m, p) for m in TRAINING_MODELS[1:]] + [sd_patch], p, rows=2)
        _training_finish(fig, axes, data, p, 'validation_loss_components')
        fig, axes = layout(3, p, columns=2, legend_rows=1, width_mm=180)
        figures.append(fig)
        for ax, model in zip(axes, ['components', *TRAINING_MODELS[3:]]):
            for metric, sid in [('initials_accuracy', 'initial'), ('finals_accuracy', 'final'), ('tones_accuracy', 'tone')]:
                _training_curve(ax, data, model, metric, p, semantic=p.registry['semantics'][sid])
            ax.set(title=p.registry['semantics'][model]['label'], ylabel='Accuracy')
        _training_legend(fig, [_training_model_handle(m, p) for m in ('initial', 'final', 'tone')] + [sd_patch], p)
        _training_finish(fig, axes, data, p, 'component_accuracy', shared_y=True)
        fig, axes = layout(1, p, columns=1, legend_rows=1, width_mm=90)
        figures.append(fig)
        _training_curve(axes[0], data, 'components', 'invalid_if_rate', p, scale=100.0)
        axes[0].set(title='Components', ylabel='Invalid predictions (%)')
        _training_legend(fig, [Line2D([], [], color=p.registry['semantics']['components']['color'], label='Validation'), sd_patch], p)
        _training_finish(fig, axes, data, p, 'invalid_if_predictions')
        return figures
    except BaseException:
        for fig in figures:
            plt.close(fig)
        raise

def write_training_report(project_root, run_id, *, summary_output, source_hashes, source_digest, protocol=None):
    if not isinstance(source_digest, str) or not re.fullmatch('[0-9a-f]{64}', source_digest):
        raise ValueError('source SHA-256 digest required')
    data, manifest = load_training_summary(summary_output)
    p = protocol or load_protocol()
    apply_style(p)
    tables = ['selected-checkpoints.md', 'selected-metrics.md', 'selected-summary.md']
    expected = {'config.json', 'data/selected-checkpoints.json', 'diagnostics/summary.json', *('tables/' + n for n in tables)}
    expected.update((f'figures/{stem}.{ext}' for stem in TRAINING_STEMS for ext in ('png', 'eps', 'metadata.json')))
    config = {'summary_manifest_sha256': file_sha256(Path(summary_output) / 'manifest.json'), 'source_digest': source_digest, 'style_sha256': p.style_hash, 'registry_sha256': p.registry_hash}
    with presentation_run(project_root, 'training-report', run_id, input_roots=[summary_output], config=config, source_hashes=source_hashes, expected_files=expected, scientific_pins=manifest['scientific_input_pins'], profile='training_report', figure_exports_on_complete='complete') as root:
        for name in tables:
            shutil.copyfile(Path(summary_output) / 'tables' / name, root / 'tables' / name)
        write_data_json(root / 'data/selected-checkpoints.json', {'selected': data['selected'], 'summary': data['selected_summary']})
        figs = build_training_figures(data, p)
        try:
            for fig, stem, fid in zip(figs, TRAINING_STEMS, TRAINING_FIGURES):
                export_figure(fig, root / 'figures', stem, fid, p, scientific_provenance={'source_digest': source_digest, 'summary_manifest_sha256': config['summary_manifest_sha256']})
        finally:
            for fig in figs:
                plt.close(fig)
        write_data_json(root / 'diagnostics/summary.json', {'figure_count': 5, 'diagnostics': data['diagnostics'], 'refit': False})
    return root

def run_training_visualization(project_root, presentation_id, *, training_runs, summary_run_id, mode, source_hashes, source_digest, protocol=None):
    if mode not in {'prepare', 'reuse'}:
        raise ValueError('mode must be prepare or reuse')
    p = protocol or load_protocol()
    apply_style(p)
    base = Path(project_root).resolve() / 'outputs'
    report = base / 'training-report' / safe_run_id(presentation_id)
    summary = base / 'training-summary' / safe_run_id(summary_run_id)
    for target in [report, summary] if mode == 'prepare' else [report]:
        ensure_output_outside_inputs(target, [v['path'] for v in training_runs.values()])
        if target.exists():
            raise FileExistsError(f'refusing to overwrite training output: {target}')
        if any(((parent / 'manifest.json').exists() for parent in target.parents)):
            raise ValueError('output is inside an existing run')
    if mode == 'prepare':
        data = load_training_runs(training_runs)
        write_training_summary(project_root, summary_run_id, data, source_hashes=source_hashes)
    else:
        data, _ = load_training_summary(summary)
        expected = {m: v['manifest_sha256'] for m, v in training_runs.items()}
        if expected != {r['model_id']: r['manifest_sha256'] for r in data['inputs']}:
            raise ValueError('reused training summary has different input pins')
    return write_training_report(project_root, presentation_id, summary_output=summary, source_hashes=source_hashes, source_digest=source_digest, protocol=p)


## 4. Verify inputs and save report
Prepare checks all histories, hashes, experiment/split identities and checkpoint selection. Test metrics never select epochs. Missing metrics remain explicitly unavailable. Reuse reads the exact saved numerical summary. Figures use the shared monochrome protocol and opaque SD bands for EPS.


In [22]:
SOURCE_HASHES = {**BUILD_METADATA["source_sha256"], **BUILD_METADATA["resource_sha256"],
    "training_visualization_builder": BUILD_METADATA["generator_sha256"],
    "embedding_builder": BUILD_METADATA["embedding_builder_sha256"]}
SOURCE_DIGEST = hashlib.sha256(json.dumps(BUILD_METADATA, sort_keys=True).encode()).hexdigest()
PROTOCOL = load_protocol()
REPORT_PATH = run_training_visualization(OUTPUT_PROJECT_ROOT, PRESENTATION_ID,
    training_runs=TRAINING_RUNS, summary_run_id=SUMMARY_RUN_ID, mode=MODE,
    source_hashes=SOURCE_HASHES, source_digest=SOURCE_DIGEST, protocol=PROTOCOL)
print("Saved training report:", REPORT_PATH)
print("Figures:", REPORT_PATH / "figures")
print("Tables:", REPORT_PATH / "tables")


Saved training report: D:\projects\Mandarin-Phonetics-Modeling\project\outputs\training-report\training-figures-01
Figures: D:\projects\Mandarin-Phonetics-Modeling\project\outputs\training-report\training-figures-01\figures
Tables: D:\projects\Mandarin-Phonetics-Modeling\project\outputs\training-report\training-figures-01\tables
